# GamaX1 Aetherion -- 200M-Param Next-Gen Base Pretraining

Is notebook mein:
1. Patched `bulk_corpus.py` install (Discord-Dialogues + Reddit-Constructive sources)
2. ~205M-param config se fresh pretraining shuru (purana 97M checkpoint overwrite NAHI hoga -- alag `out_dir`/`run_name`)
3. Crash-safe: resumable via `--resume_from`, Drive par checkpoint_interval=500 par save hota hai

**Pehle se poori ho chuki cheezein (assume kiya gaya hai):**
- Discord/Reddit ka poora conversion already `data/Discord-Dialogues/` aur `data/Reddit-Constructive/` mein maujood hai
- Patched `bulk_corpus.py` repo mein commit ho chuka hai (agar nahi, neeche step 2 mein install karo)


## 1. Drive mount + repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
if not __import__('os').path.isdir('/content/gamax1_project'):
    !git clone https://github.com/mrroy-dev/GamaX1_Aetherion.git gamax1_project
%cd /content/gamax1_project
!git pull


## 2. Patched `bulk_corpus.py` install (agar Drive/GitHub par abhi tak push nahi hua)

Agar tumne pehle hi `bulk_corpus_patched.py` ko commit+push kar diya hai (upar wale `git pull` se aa jayega),
to ye cell **skip** kar sakte ho. Agar nahi, to is cell se upload karke install karo.

In [ ]:
from google.colab import files
import shutil

# Sirf tabhi chalao jab git pull se patched version nahi aayi ho:
uploaded = files.upload()  # yahan bulk_corpus_patched.py select karo
shutil.copy("bulk_corpus_patched.py", "gamax1/bulk_corpus.py")

!git add gamax1/bulk_corpus.py
!git commit -m "Add Discord-Dialogues and Reddit-Constructive corpus sources"
!git push


## 3. Dependencies + crash-safety

`safe_disconnect.py` (agar repo mein hai) background mein chalao taaki Colab disconnect hone par
bhi training state safely flush ho jaye -- jaisa humne base-pretraining crash ke waqt use kiya tha.

In [ ]:
!pip install -r requirements.txt --quiet

import subprocess, os
if os.path.exists("safe_disconnect.py"):
    subprocess.Popen(["python3", "safe_disconnect.py"])
    print("safe_disconnect.py background mein chalu kar diya")
else:
    print("safe_disconnect.py nahi mila -- skip (zaroori nahi, bas crash-resilience helper hai)")


## 4. Sanity check -- naye sources dikh rahe hain ya nahi

Training shuru karne se pehle chhota dry-run jaisa check: data_dir mein Discord-Dialogues aur
Reddit-Constructive folder dono maujood hain, aur unme files hain.

In [ ]:
from pathlib import Path

DATA_DIR = "/content/drive/MyDrive/Aetherion_GamaX1/data"

for name in ["books_cleaned_v1", "Math_Reasoning", "Conversations-200k_clean",
             "QnA", "Discord-Dialogues", "Reddit-Constructive"]:
    p = Path(DATA_DIR) / name
    if p.is_dir():
        n_files = sum(1 for _ in p.rglob("*") if _.is_file())
        print(f"{name:28s} OK  -- {n_files:,} files")
    else:
        print(f"{name:28s} MISSING at {p}")


## 5. 200M-param fresh pretraining

Config `d_model=1024, n_heads=16, n_layers=15, n_features=4096` ~205M params deta hai
(current 97M model ke saath consistent formula se verify kiya gaya hai -- tied embeddings,
`n_features = 4 * d_model`).

Naya, ALAG `out_dir`/`experiment_dir`/`bulk_cache_dir` use ho raha hai, taaki purana 97M run
(`runs/gamax1/full/`) kabhi overwrite na ho.

**`max_steps`**: corpus ab lagbhag dugna ho gaya hai (books + math + conversations + QnA + poora
Discord + poora Reddit), isliye steps bhi usi hisaab se badhaye hain. Colab session time-limit ke
hisaab se ye resume ho sakta hai -- bas dobara yahi cell chalao, `--resume_from` khud-ba-khud
`gamax1_latest.pt` utha lega.

In [ ]:
!python3 -m gamax1.train \
    --data_dir "/content/drive/MyDrive/Aetherion_GamaX1/data" \
    --bulk_cache_dir "/content/drive/MyDrive/Aetherion_GamaX1/cache/bulk_cache_200m" \
    --tokenizer bpe \
    --bpe_vocab_size 16000 \
    --max_vocab_size 16500 \
    --d_model 1024 \
    --n_heads 16 \
    --n_layers 15 \
    --n_features 4096 \
    --block_size 128 \
    --batch_size 24 \
    --lr 3e-4 \
    --weight_decay 0.01 \
    --dropout 0.1 \
    --max_steps 220000 \
    --eval_interval 200 \
    --eval_batches 10 \
    --checkpoint_interval 500 \
    --out_dir "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1_200m/full" \
    --experiment_dir "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1_200m/experiments" \
    --run_name "gamax1_200m_v1"


### Note -- `--batch_size`

`batch_size=24` rakha hai (purane 97M run ke `32` se kam), kyunki model lagbhag dugna bada hai aur
Colab GPU memory limited hoti hai. Agar OOM (out-of-memory) error aaye, to `--batch_size` aur kam
karo (16 ya 12); agar GPU mein memory bachi dikhe, to 32 tak badha sakte ho -- training speed
behtar hogi.

## 6. (Optional, deferred) Newline-unwrap fix

Humne pehle decide kiya tha ki books ke hard-wrapped newlines ka fix `bulk_corpus.py` mein abhi
nahi daala -- kyunki uske liye poora cache rebuild chahiye. Ye run vaise bhi NAYA cache bana raha
hai (`bulk_cache_200m`), isliye agar wo fix bundle karna hai, to training shuru karne se PEHLE
`bulk_corpus.py` mein wo patch lagao. Agar skip karna hai, to ye cell khaali chhod do -- koi
nuksaan nahi, bas wahi purani known limitation continue rahegi.